In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [4]:
data_out = pd.read_pickle('results_baseline_2026-04-25_16-21-50.pkl')

In [5]:
data_out.keys()

dict_keys(['model', 'run_id', 'best_val_loss', 'test_loss', 'predictions', 'targets', 'sample_labels', 'bin_labels', 'latent_vector', 'train_losses', 'val_losses', 'val_metrics', 'latent_diagnostics', 'test_metrics'])

In [6]:
samples = np.unique(data_out['sample_labels'])

In [7]:
data = pd.read_csv("../../../data/data_merged.csv")

In [8]:
input = data[data['sample-eventid'].isin(samples)]
train_input = data[~data['sample-eventid'].isin(samples)]

In [9]:
train_input

,sample-eventid,bin_uri,occurrences,total_reads,seq,tot_repl_w_reads,kingdom,phylum,class,order,...,collection_end_date,SumExcessSpecimens,ExcessNumberTaxa,latitude,longitude,tot_repl,repl_w_reads_fractn,Excess,total_reads_per_sample,avg_reads
0,BIOPER00065,BOLD:AAA0988,0,2,AATAAATAATATAAGATTTTGACTTTTACCCCCATCATTAACTCTC...,1,Animalia,Arthropoda,Insecta,Lepidoptera,...,NaN,0.0,0.0,NaN,NaN,3,0.333333,0,684736,0.666667
1,BIOPER00065,BOLD:AAA1415,0,2,AATAAATAATATAAGATTTTGACTTTTACCCCCATCTCTAACTTTA...,1,Animalia,Arthropoda,Insecta,Lepidoptera,...,NaN,0.0,0.0,NaN,NaN,3,0.333333,0,684736,0.666667
2,BIOPER00065,BOLD:AAA1421,0,120,AATAAATAATATAAGTTTTTGACTTCTCCCCCCTTCTTTAATTCTT...,3,Animalia,Arthropoda,Insecta,Lepidoptera,...,NaN,0.0,0.0,NaN,NaN,3,1.000000,0,684736,40.000000
3,BIOPER00065,BOLD:AAA2438,0,10,AATAAATAACATAAGTTTTTGACTTTTACCACCTTCTTTAACTCTT...,2,Animalia,Arthropoda,Insecta,Lepidoptera,...,NaN,0.0,0.0,NaN,NaN,3,0.666667,0,684736,3.333333
4,BIOPER00065,BOLD:AAA5023,0,7,AATAAACAATATAAGTTTTTGACTCCTTCCCCCCTCTTTAACTCTT...,2,Animalia,Arthropoda,Insecta,Lepidoptera,...,NaN,0.0,0.0,NaN,NaN,3,0.666667,0,684736,2.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
316304,GZXAE6,BOLD:AGO4011,1,163,AATAAACAATATAAGATTTTGATTATTACCTCCTTCATTAACACTT...,2,Animalia,Arthropoda,Insecta,Diptera,...,2023-12-06,0.0,0.0,-20.794918,44.1662,3,0.666667,0,203752,54.333333
316305,GZXAE6,BOLD:AGO9189,0,3,AATAAATAACATAAGATTTTGACTATTACCCCCTTCTATCACTTTA...,1,Animalia,Arthropoda,Insecta,NaN,...,2023-12-06,0.0,0.0,-20.794918,44.1662,3,0.333333,0,203752,1.000000
316306,GZXAE6,BOLD:AGR0227,0,91,TATAAACAATATAAGATTTTGATTATTACCACCATCATTAACTCTT...,1,Animalia,Arthropoda,Insecta,NaN,...,2023-12-06,0.0,0.0,-20.794918,44.1662,3,0.333333,0,203752,30.333333
316307,GZXAE6,BOLD:AGV6048,0,4,AATAAATAACATAAGATTTTGACTATTACCCCCTTCTATAACTTTA...,1,Animalia,Arthropoda,Insecta,NaN,...,2023-12-06,0.0,0.0,-20.794918,44.1662,3,0.333333,0,203752,1.333333


In [11]:
pred = pd.DataFrame({'prediction': data_out['predictions'], 'targets': data_out['targets'], 'sample_labels': data_out['sample_labels'], 'bin_labels': data_out['bin_labels']})
pred = pd.concat([pred, pd.DataFrame(data_out['latent_vector'], columns=[f'latent_{i}' for i in range(data_out['latent_vector'].shape[1])])], axis=1)

In [14]:
all_data = pd.merge(input, pred, left_on=['sample-eventid', 'bin_uri'], right_on=['sample_labels', 'bin_labels'], how='outer').drop(columns=['sample-eventid', 'bin_uri', 'seq', 'kingdom', 'phylum', 'class']).set_index(['sample_labels', 'bin_labels'])

In [17]:
all_data.index.get_level_values('sample_labels').nunique(), all_data.index.get_level_values('bin_labels').nunique()

(515, 85522)

In [21]:
all_data

occurrences  total_reads  tot_repl_w_reads  \
sample_labels bin_labels                                                 
BIOPER00138   BOLD:AAA0178            0           14                 3   
              BOLD:AAA0424            0            6                 1   
              BOLD:AAA0988            0            5                 1   
              BOLD:AAA0997            0           10                 2   
              BOLD:AAA1328            0            4                 2   
...                                 ...          ...               ...   
GZYSTR        BOLD:AGT3966            2         4704                 3   
              BOLD:AGV0195            0            5                 1   
              BOLD:AGW8153            0           14                 3   
              BOLD:AGW9919            0            4                 1   
              BOLD:AGZ2097            0           21                 3   

                                  order          family        genus  \
sample_labels bin_labels                                               
BIOPER00138   BOLD:AAA0178  Lepidoptera     Gelechiidae   Dichomeris   
              BOLD:AAA0424          NaN             NaN          NaN   
              BOLD:AAA0988  Lepidoptera     Gelechiidae        Sinoe   
              BOLD:AAA0997  Lepidoptera     Gelechiidae          NaN   
              BOLD:AAA1328  Lepidoptera        Erebidae  Leucotmemis   
...                                 ...             ...          ...   
GZYSTR        BOLD:AGT3966      Diptera  Dolichopodidae          NaN   
              BOLD:AGV0195          NaN             NaN          NaN   
              BOLD:AGW8153      Diptera   Cecidomyiidae          NaN   
              BOLD:AGW9919  Hymenoptera      Braconidae          NaN   
              BOLD:AGZ2097  Hymenoptera          Apidae          NaN   

                                             species  min_reads  max_reads  \
sample_labels bin_labels                                                     
BIOPER00138   BOLD:AAA0178  Dichomeris santarosensis          3          6   
              BOLD:AAA0424                       NaN          0          6   
              BOLD:AAA0988                       NaN          0          5   
              BOLD:AAA0997                       NaN          0          5   
              BOLD:AAA1328          Leucotmemis nexa          0          2   
...                                              ...        ...        ...   
GZYSTR        BOLD:AGT3966                       NaN       1323       1821   
              BOLD:AGV0195                       NaN          0          5   
              BOLD:AGW8153                       NaN          2          8   
              BOLD:AGW9919                       NaN          0          4   
              BOLD:AGZ2097                       NaN          3         13   

                              sd_reads  ...  latent_0  latent_1  latent_2  \
sample_labels bin_labels                ...                                 
BIOPER00138   BOLD:AAA0178    1.527525  ...  0.073160 -0.006914 -0.023308   
              BOLD:AAA0424    3.464102  ... -0.000449  0.000300  0.000113   
              BOLD:AAA0988    2.886751  ... -0.088616  0.030118  0.008262   
              BOLD:AAA0997    2.886751  ... -0.015243  0.000848  0.000704   
              BOLD:AAA1328    1.154701  ... -0.000271 -0.000337 -0.000295   
...                                ...  ...       ...       ...       ...   
GZYSTR        BOLD:AGT3966  249.096367  ...       NaN       NaN       NaN   
              BOLD:AGV0195    2.886751  ...       NaN       NaN       NaN   
              BOLD:AGW8153    3.055050  ...       NaN       NaN       NaN   
              BOLD:AGW9919    2.309401  ...       NaN       NaN       NaN   
              BOLD:AGZ2097    5.291503  ...       NaN       NaN       NaN   

                            latent_3  latent_4  latent_5  latent_6  latent_7  \
sample_labels bin_labels       

In [22]:
all_data[all_data['prediction'] != 0]

,,occurrences,total_reads,tot_repl_w_reads,order,family,genus,species,min_reads,max_reads,sd_reads,...,latent_0,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,latent_7,latent_8,latent_9
sample_labels,bin_labels,,,,,,,,,,,,,,,,,,,,,
BIOPER00138,BOLD:AGE5680,0,9,3,Lepidoptera,Gelechiidae,NaN,NaN,2,4,1.000000,...,-0.000258,0.000042,-0.000017,0.000013,-0.000036,0.000002,0.000012,-0.000134,-0.000048,0.000460
G16R5R,BOLD:AFV0053,8,254,3,Diptera,Sciaridae,NaN,NaN,56,115,29.535290,...,-0.000057,-0.000078,-0.000079,-0.000094,0.000062,-0.000097,0.000084,0.000010,0.000047,0.000209
G17O2Y,BOLD:AFA1866,0,2,1,Diptera,Chironomidae,NaN,NaN,0,2,1.154701,...,0.001245,-0.000515,-0.000278,-0.000126,0.000551,-0.000149,0.000564,0.000975,0.000475,-0.001573
G17RAD,BOLD:AAE7838,0,8,1,Hymenoptera,Formicidae,Camponotus,Camponotus tendryi,0,8,4.618802,...,0.000011,-0.000224,-0.000078,-0.000077,0.000089,-0.000075,0.000046,0.000054,0.000081,0.000088
G19AZB,BOLD:ABA8270,3,195,3,Diptera,Sciaridae,Hyperlasion,NaN,26,103,38.509739,...,-0.001057,0.000369,0.000086,0.000008,-0.000274,0.000049,-0.000256,-0.000675,-0.000191,0.001586
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GZPO2W,BOLD:ABA8270,3,89,3,Diptera,Sciaridae,Hyperlasion,NaN,24,35,5.507571,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GZSO1T,BOLD:AGG9073,0,19,1,Diptera,Ceratopogonidae,NaN,NaN,0,19,10.969655,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GZST6D,BOLD:AAD3923,7,58,3,Hymenoptera,Formicidae,Pheidole,Pheidole spinosa,4,48,24.846194,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
print("Prediction statistics:")
print(f"Min: {pred['prediction'].min()}")
print(f"Max: {pred['prediction'].max()}")
print(f"Mean: {pred['prediction'].mean()}")
print(f"Median: {pred['prediction'].median()}")
print(f"Std: {pred['prediction'].std()}")
print(f"\nValue counts:")
print(pred['prediction'].value_counts().head(20))
print(f"\nUnique values: {pred['prediction'].nunique()}")
print(f"\nNumber of zeros: {(pred['prediction'] == 0).sum()}")
print(f"Number of ones: {(pred['prediction'] == 1).sum()}")
print(f"\nDistribution per sample:")
for sample in pred['sample_labels'].unique()[:5]:
    sample_preds = pred[pred['sample_labels'] == sample]['prediction']
    print(f"  {sample}: {len(sample_preds)} predictions, {(sample_preds == 0).sum()} zeros, {(sample_preds == 1).sum()} ones")

Prediction statistics:
Min: 0.0
Max: 1.0
Mean: 0.0024525560438632965
Median: 0.0
Std: 0.049462638795375824

Value counts:
prediction
0.0    209470
1.0       515
Name: count, dtype: int64

Unique values: 2

Number of zeros: 209470
Number of ones: 515

Distribution per sample:
  BIOPER00138: 389 predictions, 388 zeros, 1 ones
  G16R5R: 291 predictions, 290 zeros, 1 ones
  G17O2Y: 739 predictions, 738 zeros, 1 ones
  G17RAD: 229 predictions, 228 zeros, 1 ones
  G19AZB: 370 predictions, 369 zeros, 1 ones


In [24]:
print("\nTarget statistics:")
print(f"Min: {pred['targets'].min()}")
print(f"Max: {pred['targets'].max()}")
print(f"Mean: {pred['targets'].mean()}")
print(f"Median: {pred['targets'].median()}")
print(f"Std: {pred['targets'].std()}")
print(f"\nTarget value counts (first 20):")
print(pred['targets'].value_counts().head(20))
print(f"\nUnique target values: {pred['targets'].nunique()}")
print(f"\nNumber of zero targets: {(pred['targets'] == 0).sum()}")
print(f"Number of one targets: {(pred['targets'] == 1).sum()}")

print("\n\nSample-level targets (per sample, sum should be ~1):")
for sample in pred['sample_labels'].unique()[:5]:
    sample_targets = pred[pred['sample_labels'] == sample]['targets']
    print(f"  {sample}: sum={sample_targets.sum():.6f}, n_bins={len(sample_targets)}, values={sample_targets.unique()[:5]}")


Target statistics:
Min: 0.0
Max: 0.8949704170227051
Mean: 0.002447793958708644
Median: 0.0003516174328979105
Std: 0.008953611366450787

Target value counts (first 20):
targets
0.000000    103448
0.000536       675
0.000702       566
0.000563       548
0.000433       536
0.000479       514
0.000150       501
0.000763       491
0.000460       483
0.000912       476
0.000745       472
0.000710       460
0.000853       439
0.000959       433
0.000891       431
0.000446       429
0.000826       424
0.001488       417
0.002976       409
0.000794       401
Name: count, dtype: int64

Unique target values: 5129

Number of zero targets: 103448
Number of one targets: 0


Sample-level targets (per sample, sum should be ~1):
  BIOPER00138: sum=0.000000, n_bins=389, values=[0.]
  G16R5R: sum=1.000000, n_bins=291, values=[0.00619195 0.         0.0123839  0.01547988 0.0247678 ]
  G17O2Y: sum=1.000000, n_bins=739, values=[0.         0.00085985 0.00171969 0.00257954 0.01203783]
  G17RAD: sum=1.000000, 

In [25]:
print("\nDetailed sample analysis:")
sample_sums = pred.groupby('sample_labels')['targets'].sum()
print(f"Samples with zero total targets (no observations): {(sample_sums == 0).sum()}")
print(f"Samples with non-zero total targets: {(sample_sums > 0).sum()}")
print(f"\nZero-target samples (first 20):")
zero_samples = sample_sums[sample_sums == 0].index[:20].tolist()
for sample in zero_samples:
    sample_data = pred[pred['sample_labels'] == sample]
    print(f"  {sample}: {len(sample_data)} BINs, all targets={sample_data['targets'].unique().tolist()}")

print(f"\n\nCross-check: predictions vs targets per sample")
print("\nFor samples with ZERO targets:")
for sample in zero_samples[:3]:
    sample_data = pred[pred['sample_labels'] == sample]
    print(f"  {sample}:")
    print(f"    Predictions: {sample_data['prediction'].value_counts().to_dict()}")
    print(f"    All targets are: {sample_data['targets'].unique().tolist()}")

print("\nFor samples with NON-ZERO targets:")
non_zero_samples = sample_sums[sample_sums > 0].index[:3].tolist()
for sample in non_zero_samples:
    sample_data = pred[pred['sample_labels'] == sample]
    print(f"  {sample}:")
    print(f"    Predictions: min={sample_data['prediction'].min():.6f}, max={sample_data['prediction'].max():.6f}, sum={sample_data['prediction'].sum():.6f}")
    print(f"    Targets: min={sample_data['targets'].min():.6f}, max={sample_data['targets'].max():.6f}, sum={sample_data['targets'].sum():.6f}")


Detailed sample analysis:
Samples with zero total targets (no observations): 1
Samples with non-zero total targets: 514

Zero-target samples (first 20):
  BIOPER00138: 389 BINs, all targets=[0.0]


Cross-check: predictions vs targets per sample

For samples with ZERO targets:
  BIOPER00138:
    Predictions: {0.0: 388, 1.0: 1}
    All targets are: [0.0]

For samples with NON-ZERO targets:
  G16R5R:
    Predictions: min=0.000000, max=1.000000, sum=1.000000
    Targets: min=0.000000, max=0.130031, sum=1.000000
  G17O2Y:
    Predictions: min=0.000000, max=1.000000, sum=1.000000
    Targets: min=0.000000, max=0.135856, sum=1.000000
  G17RAD:
    Predictions: min=0.000000, max=1.000000, sum=1.000000
    Targets: min=0.000000, max=0.057803, sum=1.000000


In [26]:
import torch
import torch.nn.functional as F
import numpy as np

print("\n\n=== DEMONSTRATING THE PROBLEM ===\n")

# Simulate what happens with cross-entropy loss on all-zero targets
print("1. Cross-entropy loss with zero targets:")
print("-" * 50)

# Example: 5 BINs, all targets are 0
targets = torch.tensor([[0.0, 0.0, 0.0, 0.0, 0.0]])
print(f"Targets: {targets.numpy()}")

# Try different logits outputs
logits_cases = [
    ("All equal logits", torch.tensor([[-1.0, -1.0, -1.0, -1.0, -1.0]])),
    ("Concentrated logits", torch.tensor([[10.0, -10.0, -10.0, -10.0, -10.0]])),
    ("Random logits", torch.tensor([[0.5, -0.2, 0.3, -0.1, 0.2]])),
]

for name, logits in logits_cases:
    log_probs = F.log_softmax(logits, dim=-1)
    probs = F.softmax(logits, dim=-1)
    loss = -(targets * log_probs).sum(dim=-1)
    
    print(f"\n{name}:")
    print(f"  Logits: {logits.numpy()}")
    print(f"  Softmax probs: {probs.numpy()}")
    print(f"  Cross-entropy loss: {loss.item():.6f}")
    
print("\n" + "="*50)
print("\nKEY INSIGHT:")
print("When ALL targets are 0, the cross-entropy loss is ALWAYS 0 regardless of predictions!")
print("This means the model receives NO GRADIENT SIGNAL about what to predict.")
print("With softmax constraint (must sum to 1), model defaults to one-hot-like outputs.")

# Verify this with the actual test data
print("\n\nVerifying with actual test data:")
print("-" * 50)
sample_with_all_zeros = 'BIOPER00138'
sample_with_values = 'G16R5R'

print(f"\nSample {sample_with_all_zeros} (all-zero targets):")
targets_zero = pred[pred['sample_labels'] == sample_with_all_zeros]['targets'].values
preds_zero = pred[pred['sample_labels'] == sample_with_all_zeros]['prediction'].values
print(f"  Targets (first 10): {targets_zero[:10]}")
print(f"  Predictions (first 10): {preds_zero[:10]}")
print(f"  Target sum: {targets_zero.sum()}")
print(f"  Pred sum: {preds_zero.sum()}")

print(f"\nSample {sample_with_values} (normal targets):")
targets_normal = pred[pred['sample_labels'] == sample_with_values]['targets'].values
preds_normal = pred[pred['sample_labels'] == sample_with_values]['prediction'].values
print(f"  Targets (first 10): {targets_normal[:10]}")
print(f"  Predictions (first 10): {preds_normal[:10]}")
print(f"  Target sum: {targets_normal.sum():.6f}")
print(f"  Pred sum: {preds_normal.sum():.6f}")



=== DEMONSTRATING THE PROBLEM ===

1. Cross-entropy loss with zero targets:
--------------------------------------------------
Targets: [[0. 0. 0. 0. 0.]]

All equal logits:
  Logits: [[-1. -1. -1. -1. -1.]]
  Softmax probs: [[0.2 0.2 0.2 0.2 0.2]]
  Cross-entropy loss: -0.000000

Concentrated logits:
  Logits: [[ 10. -10. -10. -10. -10.]]
  Softmax probs: [[1.0000000e+00 2.0611537e-09 2.0611537e-09 2.0611537e-09 2.0611537e-09]]
  Cross-entropy loss: -0.000000

Random logits:
  Logits: [[ 0.5 -0.2  0.3 -0.1  0.2]]
  Softmax probs: [[0.27739665 0.1377511  0.22711317 0.1522385  0.20550048]]
  Cross-entropy loss: -0.000000


KEY INSIGHT:
When ALL targets are 0, the cross-entropy loss is ALWAYS 0 regardless of predictions!
This means the model receives NO GRADIENT SIGNAL about what to predict.
With softmax constraint (must sum to 1), model defaults to one-hot-like outputs.


Verifying with actual test data:
--------------------------------------------------

Sample BIOPER00138 (all-zero 

## ROOT CAUSE ANALYSIS: Why Predictions Are All 0s and 1s

### The Problem
Your model outputs **one-hot-like predictions** (exactly 0 or 1, with one BIN per sample getting 1.0) instead of relative abundance distributions. This happens because **excess samples have NO observed occurrences** in your original data.

### Why This Happens (3-Step Breakdown)

#### Step 1: Excess Samples Have Zero Occurrences
In `data_merged.csv`, samples marked with `Excess=1` have `occurrences=0` for ALL BINs:
- These represent samples where BINs were too numerous to count individually
- Only count and taxonomic class of excess were recorded
- No actual occurrence counts to supervise on

#### Step 2: Cross-Entropy Loss Is Indifferent to Zero Targets
Your loss function: `loss = -sum(target * log(prob))`

With `target = 0` for all BINs:
- `loss = -sum(0 * log(prob)) = 0` **regardless of predictions**
- The model receives **NO GRADIENT SIGNAL**
- No learning happens for these samples

#### Step 3: Softmax Without Supervision Defaults to One-Hot
Since softmax enforces `sum(probs) = 1` but gets no loss signal:
- Model has no incentive to match target distribution (it's all zeros!)
- Model defaults to one-hot based on latent vectors or initialization
- Each sample gets exactly one BIN with `prob ≈ 1.0`, rest get `prob ≈ 0.0`

### Why Other Samples Work Correctly
Non-excess samples with real occurrence counts:
- Have non-zero targets summing to 1.0
- Cross-entropy loss provides clear gradients
- Model learns meaningful probability distributions
- Predictions are continuous values, not one-hot

### The Fundamental Problem
Your training strategy assumes:
1. ✓ Train on non-excess samples (good relative abundance data)
2. ✓ Test on excess samples (to measure over-estimation)
3. ✗ **BUT: Excess samples provide zero targets = no supervision!**

The model never learns what to predict when `target=[0,0,0,...,0]` because the loss is always zero. This isn't the model "learning wrong" — it's the model receiving no information at all.

### Solutions

#### Option 1: Use Regularization During Inference Only ❌
Would help but doesn't solve the core issue of zero targets during training.

#### Option 2: Modify Loss for Zero-Target Samples ⚠️
- Could add a regularization term to push towards uniform distribution for zero-target samples
- But this contradicts your goal: excess samples should be "over-estimated" by the model
- Would require careful tuning to not harm non-excess samples

#### Option 3: Restructure the Training Problem ✅ **RECOMMENDED**
**Change what "excess samples" mean in your pipeline:**

Instead of `occurrences=0`:
- **Option 3a:** Assign excess occurrences to BINs proportionally based on test set prevalence
  - Run a forward pass on all observed BINs
  - Distribute excess proportionally to their relative abundances
  - Creates realistic training signal

- **Option 3b:** Use a separate "binary classification" approach for excess samples
  - Question: "Was this BIN present in excess?" (present/absent)
  - Train logistic regression instead of abundance prediction
  - Separate pipeline: abundance model + presence classifier

- **Option 3c:** Use semi-supervised learning
  - Treat excess samples as unlabeled data
  - Use consistency regularization or pseudo-labeling
  - More sophisticated but might work

#### Option 4: Accept One-Hot and Use Differently ⚠️
If you want to identify which BINs were likely in excess:
- Current one-hot predictions DO identify "the model's best guess"
- Could use the "chosen" BIN as your excess-BIN candidate
- But loses information about relative abundances of multiple excess BINs

### Recommendations
1. **Investigate the excess data more:** Can you assign actual occurrence counts rather than zeros?
2. **Test Option 3a:** Proportional redistribution is most principled and maintains your original goal
3. **Monitor training:** Add loss diagnostics to catch when samples provide zero signal
4. **Consider the interpretation:** What do you actually want to predict for excess samples?